# Assignment 1

a) 
- P is the set of all percepts the robot can recieve. For example pt = [location, obstacle_nearby, battery_level, package_status]
- P* is the set of all possible sequences of percepts the robot could have seen so far. 
- A is -the set of possible actions. 
- f is the agents decision rule which maps percept history to the action the robot chooses

b)
A performance measure could be the cumulative reward points the robot gets. For example, the sum of r_t from t=1 to T, where r_t is the reward recieved during step t

c)
4 moves: -20
1 pickup: +10
1 delivery: +50
6 total steps = -6
total: 34

d) This is most likely a utility based agent because it uses the reward values to decide which actions are preferable.

e) The real world introduces uncertaintly. Sensors can be inaccurate, physics are not ideal, things can break unexepctedly. This might cause one of the actions does not execute properly or not be recorded properly. 


In [15]:
GRID_SIZE = 5

package_location = (2, 2)
delivery_location = (4, 4)

#place obstacles manually
obstacles = [(1, 1), (3, 2)]
#obstacles = []

#robot state
robot = {
    "location": (0, 0),
    "battery": 10,
    "carrying_package": False,
    "delivered": False
}

# Metrics for this run
robot_path = [robot["location"]]
actions_taken = []


def choose_action(robot):
    x, y = robot["location"]

    #actions
    if robot["location"] == package_location and not robot["carrying_package"]:
        return "pick_up"

    if robot["location"] == delivery_location and robot["carrying_package"]:
        return "deliver"

    if robot["carrying_package"]:
        target_x, target_y = delivery_location
    else:
        target_x, target_y = package_location

    # simple movements
    if x < target_x:
        return "move_down"
    elif x > target_x:
        return "move_up"
    elif y < target_y:
        return "move_right"
    elif y > target_y:
        return "move_left"

    return "wait"


def perform_action(robot, action):
    x, y = robot["location"]

    if action == "pick_up":
        robot["carrying_package"] = True
        print("Package picked up!")

    elif action == "deliver":
        robot["carrying_package"] = False
        robot["delivered"] = True
        print("Package delivered!")

    else:
        new_x, new_y = x, y

        if action == "move_up":
            new_x -= 1
        elif action == "move_down":
            new_x += 1
        elif action == "move_left":
            new_y -= 1
        elif action == "move_right":
            new_y += 1

        new_location = (new_x, new_y)

        def can_move(location):
            row, column = location
            return (
                0 <= row < GRID_SIZE
                and 0 <= column < GRID_SIZE
                and location not in obstacles
            )

        if can_move(new_location):
            robot["location"] = new_location
        else:
            print("Movement blocked; trying to go around")

            # If horizontal movement is blocked, try up or down.
            # If vertical movement is blocked, try left or right.
            if action in ("move_left", "move_right"):
                alternatives = [(x - 1, y), (x + 1, y)]
            else:
                alternatives = [(x, y - 1), (x, y + 1)]

            for alternative in alternatives:
                if can_move(alternative):
                    robot["location"] = alternative
                    print("Moved around obstacle to:", alternative)
                    break
            else:
                print("No open space to move around obstacle")

    robot["battery"] -= 1


# Run simulation
while robot["battery"] > 0 and not robot["delivered"]:

    print("\nRobot:", robot)

    action = choose_action(robot)
    actions_taken.append(action)

    print("Action:", action)

    perform_action(robot, action)

    # Record each coordinate the robot moves through.
    if robot["location"] != robot_path[-1]:
        robot_path.append(robot["location"])


if robot["delivered"]:
    status = "SUCCESS"
    print("\nDelivery successful!")
else:
    status = "FAILURE"
    print("\nBattery died before delivery.")

print("\nRun Metrics")
print("Robot path:", robot_path)
print("Actions:", actions_taken)
print("Final battery:", robot["battery"])
print("Status:", status)


Robot: {'location': (0, 0), 'battery': 10, 'carrying_package': False, 'delivered': False}
Action: move_down

Robot: {'location': (1, 0), 'battery': 9, 'carrying_package': False, 'delivered': False}
Action: move_down

Robot: {'location': (2, 0), 'battery': 8, 'carrying_package': False, 'delivered': False}
Action: move_right

Robot: {'location': (2, 1), 'battery': 7, 'carrying_package': False, 'delivered': False}
Action: move_right

Robot: {'location': (2, 2), 'battery': 6, 'carrying_package': False, 'delivered': False}
Action: pick_up
Package picked up!

Robot: {'location': (2, 2), 'battery': 5, 'carrying_package': True, 'delivered': False}
Action: move_down
Movement blocked; trying to go around
Moved around obstacle to: (2, 1)

Robot: {'location': (2, 1), 'battery': 4, 'carrying_package': True, 'delivered': False}
Action: move_down

Robot: {'location': (3, 1), 'battery': 3, 'carrying_package': True, 'delivered': False}
Action: move_down

Robot: {'location': (4, 1), 'battery': 2, 'carr

Notes: Moving greedily towards a target can cause it to get stuck on obstacles and die. Some obstaces resolution logic needs to be implemented to navigate around them.

We can resolve obstacles by the following: if the obstacle is above or below, we can move left and right by one and then retry the initial movement. We can do the same for an obstacle to the left or right.

We can use battery as score, with higher battery being better.
Scenario 1 (No Obstacles) : 
 - Robot path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
 - Actions: ['move_down', 'move_down', 'move_right', 'move_right', 'pick_up', 'move_down', 'move_down', 'move_right', 'move_right', 'deliver']
 - battery: 10
 - success


Scenario 2 (Few Obstacles): 
- Robot: {'location': (2, 1), 'battery': 14, 'carrying_package': True, 'delivered': False}
- Action: move_down
...
- Robot path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]
- Actions: ['move_down', 'move_down', 'move_right', 'move_right', 'pick_up', 'move_down', 'move_down', 'move_down', 'move_right', 'move_right', 'move_right', 'deliver']
- Final battery: 8
- Status: SUCCESS

Scenario 3 (Low Battery: 10)  : 
- Robot path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3)]
- Actions: ['move_down', 'move_down', 'move_right', 'move_right', 'pick_up', 'move_down', 'move_down', 'move_down', 'move_right', 'move_right']
- Final battery: 0
- Status: FAILURE


Scenario 1 with no obstacles achieves the highest score of 10 remaining. Adding obstacles expends 2 more battery life. Scenario 3, starting with 10 battery, fails.
